# Part A — Waves in a box

**Working time:** about 3 hours  
**Work in groups of 2–4.** This notebook is guided and is not submitted.

## Learning goals

By the end you should be able to:

- configure and run the shallow-water model;
- predict and measure the long-wave speed (c=\sqrt{gH});
- identify an incident and reflected wave;
- carry out a controlled comparison in which only one parameter changes.

Before beginning, write down one sentence describing what you expect a
localized elevation of the water surface to do in a closed basin.


**Prediction 1.** What will a localized elevation do after the model starts?

**Your response:**


## 1. Installation and imports

The course uses the released PyPI package, not a source checkout. Install
the standard version in a terminal with:

```text
python -m pip install "shallowwater==0.1.4"
```

Optional acceleration is installed with:

```text
python -m pip install "shallowwater[numba]==0.1.4"
```

The first numba-backed run can pause while functions are compiled. Both
backends solve the same discrete equations.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from shallowwater import (
    ModelParams, backend_info, compute_dt_cfl, depth_on_u,
    make_grid, run_model, zero_forcing,
)

print(backend_info())


## 2. A controlled right-going pulse

The model variables are surface displacement (eta) and depth-averaged
velocities (u,v). The initial velocity below is chosen so most of the
disturbance travels toward increasing (x), which makes speed and
reflection easier to measure.

The model uses solid vertical walls. It does not include wave breaking,
wetting and drying, or coastal inundation.


In [ ]:
def cross_basin_pulse(grid, params, *, amplitude=0.10, radius=60e3, x0=300e3):
    eta_line = amplitude * np.exp(-((grid.x_c - x0) / radius) ** 2)
    eta = np.repeat(eta_line[None, :], grid.Ny, axis=0)

    H_u = depth_on_u(grid, params.H)
    eta_u = amplitude * np.exp(-((grid.x_u - x0) / radius) ** 2)
    u = np.repeat(eta_u[None, :], grid.Ny, axis=0) * np.sqrt(params.g / H_u)
    v = np.zeros((grid.Ny + 1, grid.Nx))
    return eta, u, v


def run_uniform_case(*, H=400.0, Lx=1.6e6, Nx=160, tmax_hours=8.0):
    Ny, Ly = 20, 200e3
    grid = make_grid(Nx, Ny, Lx, Ly)
    params = ModelParams(H=H, g=9.81, f0=0.0, beta=0.0, r=0.0, linear=True)
    dt = compute_dt_cfl(grid, params, cfl=0.45)
    out = run_model(
        tmax=tmax_hours * 3600,
        dt=dt,
        grid=grid,
        params=params,
        forcing_fn=zero_forcing,
        ic_fn=lambda g, p: cross_basin_pulse(g, p),
        save_every=4,
        out_vars=("eta", "u"),
    )
    print(
        f"H={H:.0f} m, Lx={Lx/1e3:.0f} km, dx={grid.dx/1e3:.1f} km, "
        f"dt={dt:.1f} s, saved={len(out['time'])}"
    )
    return grid, params, out


In [ ]:
grid, params, out = run_uniform_case()
eta = np.asarray(out["eta"])
times = np.asarray(out["time"])
eta_line = eta.mean(axis=1)

fig, ax = plt.subplots(figsize=(9, 4))
image = ax.pcolormesh(grid.x_c / 1e3, times / 3600, eta_line, shading="auto", cmap="RdBu_r")
ax.set(xlabel="x [km]", ylabel="time [hours]", title="Centreline Hovmöller diagram")
fig.colorbar(image, ax=ax, label="surface displacement [m]")
plt.show()


**Observation 1.** Identify the incident and reflected branches in the Hovmöller diagram. What happens at the eastern wall?

**Your response:**


## 3. Predict and measure wave speed

For a uniform-depth shallow-water wave,

[
c_{theory}=\sqrt{gH}.
]

We measure the position of the maximum before the pulse reaches the wall.
A grid introduces uncertainty of roughly one grid cell in position.


In [ ]:
c_theory = np.sqrt(params.g * float(params.H))
target_time = 2.5 * 3600
time_index = int(np.argmin(np.abs(times - target_time)))
x_initial = 300e3
x_peak = grid.x_c[np.argmax(eta_line[time_index])]
c_measured = (x_peak - x_initial) / times[time_index]
relative_error = abs(c_measured - c_theory) / c_theory

print(f"theoretical speed = {c_theory:.2f} m/s")
print(f"measured speed    = {c_measured:.2f} m/s")
print(f"relative error    = {100*relative_error:.1f} %")


**Analysis 1.** Report the theoretical and measured speeds. Are they consistent given the grid spacing?

**Your response:**


## 4. Controlled depth experiment

Change only the depth from 400 m to 900 m. Predict the speed ratio before
running. Keep the domain and initial disturbance unchanged.


**Prediction 2.** What is (c_{900}/c_{400})? Will reflection occur earlier or later?

**Your response:**


In [ ]:
grid_deep, params_deep, out_deep = run_uniform_case(H=900.0, tmax_hours=5.5)
eta_deep = np.asarray(out_deep["eta"]).mean(axis=1)
times_deep = np.asarray(out_deep["time"])

fig, ax = plt.subplots(figsize=(9, 4))
image = ax.pcolormesh(
    grid_deep.x_c / 1e3, times_deep / 3600, eta_deep,
    shading="auto", cmap="RdBu_r"
)
ax.set(xlabel="x [km]", ylabel="time [hours]", title="H = 900 m")
fig.colorbar(image, ax=ax, label="surface displacement [m]")
plt.show()


**Analysis 2.** Does the numerical comparison support the predicted depth scaling? Give evidence from the plot or a measurement.

**Your response:**


## 5. Optional investigations

If time permits, try one of these while changing only one primary factor:

- Double the basin length. How does the wall-arrival time change?
- Move the initial pulse. Which arrival times change and which speeds do not?
- Change the pulse amplitude. Does speed change in the linear model?
- Change `Nx` while keeping `Lx` fixed. How does numerical error change?

## Checkpoint

Save a short record of the parameter changed, prediction, observation,
quantitative evidence, and one model limitation.


**Final reflection.** Name one physical conclusion and one limitation of this experiment.

**Your response:**
